In [3]:
import pandas as pd
import numpy as np

# Load datasets
fed_cycle = pd.read_csv('D:/SHE CARES/FedCycleData071012.csv')
menstrual_cycle = pd.read_csv('D:/SHE CARES/Menstural_cyclelength.csv')
pcos_without_infertility = pd.read_excel('D:/SHE CARES/PCOS_data_without_infertility.xlsx')
pcos_infertility = pd.read_csv('D:/SHE CARES/PCOS_infertility.csv')
maternal_health = pd.read_csv('D:/SHE CARES/Maternal Health Risk Data Set.csv')
pcos_data = pd.read_csv('D:/SHE CARES/PCOS_data.csv')  # Added dataset

# Function to convert age to bin
def age_to_bin(age):
    try:
        age_int = int(age)
    except:
        try:
            age_int = int(str(age).split('-')[0])
        except:
            return np.nan
    if age_int < 20:
        return 15
    elif age_int <= 25:
        return 20
    elif age_int <= 30:
        return 25
    elif age_int <= 35:
        return 30
    elif age_int <= 40:
        return 35
    elif age_int <= 45:
        return 40
    elif age_int <= 50:
        return 45
    else:
        return 50

# Apply age binning
fed_cycle['age_bin'] = fed_cycle['Age'].apply(age_to_bin)
menstrual_cycle['age_bin'] = menstrual_cycle['age'].apply(age_to_bin)
pcos_without_infertility['age_bin'] = pcos_without_infertility['Age'].apply(age_to_bin)
maternal_health['age_bin'] = maternal_health['Age'].apply(age_to_bin)
pcos_data['age_bin'] = pcos_data[' Age (yrs)'].apply(age_to_bin)  # Age column name in PCOS_data

# Specify the expected date format and parse dates explicitly
menstrual_cycle['cycle_start_date'] = pd.to_datetime(
    menstrual_cycle['cycle_start_date'], format='%m/%d/%y', errors='coerce')
menstrual_cycle['cycle_end_date'] = pd.to_datetime(
    menstrual_cycle['cycle_end_date'], format='%m/%d/%y', errors='coerce')

# Forward fill missing values properly
fed_cycle.ffill(inplace=True)
menstrual_cycle.ffill(inplace=True)
pcos_without_infertility.ffill(inplace=True)
maternal_health.ffill(inplace=True)
pcos_data.ffill(inplace=True)  # Fill missing values for PCOS_data

# Encoding categorical symptom data in PCOS dataset without infertility
yes_no_map = {'Yes': 1, 'No': 0, 'Sometimes': 0.5, 'Two or more days a week': 1}
for col in pcos_without_infertility.columns:
    if pcos_without_infertility[col].dtype == 'object' and col not in ['Timestamp', 'Age']:
        pcos_without_infertility[col] = pcos_without_infertility[col].map(yes_no_map).fillna(0)

# Encoding categorical data in new PCOS_data dataset
bool_cols_pcos_data = ['Pregnant(Y/N)', 'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)',
                      'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)', 'Reg.Exercise(Y/N)']
for col in bool_cols_pcos_data:
    if col in pcos_data.columns:
        pcos_data[col] = pcos_data[col].apply(lambda x: 1 if str(x).strip().upper() in ['YES', 'Y', '1'] else 0)

# Rename PCOS infertility patient id column for consistency
if 'Patient File No.' in pcos_infertility.columns:
    pcos_infertility.rename(columns={'Patient File No.': 'PatientID'}, inplace=True)

print("Cleaning and preparation complete. Datasets are ready for merging or further analysis.")


Cleaning and preparation complete. Datasets are ready for merging or further analysis.


In [5]:
import pandas as pd
import numpy as np

# Define helper to aggregate numeric columns grouped by age_bin
def agg_numeric_only(df):
    numeric_cols = df.select_dtypes(include='number').columns
    return df[['age_bin'] + list(numeric_cols.drop('age_bin', errors='ignore'))].groupby('age_bin').mean().reset_index()

# Age binning function
def age_to_bin(age):
    try:
        age_int = int(age)
    except:
        try:
            age_int = int(str(age).split('-')[0])
        except:
            return np.nan
    if age_int < 20:
        return 15
    elif age_int <= 25:
        return 20
    elif age_int <= 30:
        return 25
    elif age_int <= 35:
        return 30
    elif age_int <= 40:
        return 35
    elif age_int <= 45:
        return 40
    elif age_int <= 50:
        return 45
    else:
        return 50

# Load cleaned datasets
fed_cycle = pd.read_csv('fed_cycle_cleaned.csv')
menstrual_cycle = pd.read_csv('menstrual_cycle_cleaned.csv')
pcos_no_infert = pd.read_csv('pcos_without_infertility_cleaned.csv')
maternal_health = pd.read_csv('maternal_health_cleaned.csv')
pcos_data = pd.read_csv('PCOS_data.csv')

# Add age_bin to PCOS_data before aggregation
pcos_data['age_bin'] = pcos_data[' Age (yrs)'].apply(age_to_bin)

# Aggregate numeric columns only
fed_agg = agg_numeric_only(fed_cycle)
menstrual_agg = agg_numeric_only(menstrual_cycle)
maternal_agg = agg_numeric_only(maternal_health)
pcos_no_infert_agg = agg_numeric_only(pcos_no_infert)
pcos_data_agg = agg_numeric_only(pcos_data)

# Merge aggregated DataFrames on age_bin
merged_agg = pd.merge(fed_agg, menstrual_agg, on='age_bin', how='outer', suffixes=('_fed', '_menstrual'))
merged_agg = pd.merge(merged_agg, maternal_agg, on='age_bin', how='outer')
merged_agg = pd.merge(merged_agg, pcos_no_infert_agg, on='age_bin', how='outer', suffixes=('', '_pcos_no_infert'))
merged_agg = pd.merge(merged_agg, pcos_data_agg, on='age_bin', how='outer', suffixes=('', '_pcos_data'))

# Save merged aggregated dataset
merged_agg.to_csv('merged_aggregated_womens_health_with_pcos.csv', index=False)

print("Merged aggregated dataset with PCOS_data saved as 'merged_aggregated_womens_health_with_pcos.csv'.")
print(merged_agg.head())
print('Datasets merged approximately by age_bin and saved.')


Merged aggregated dataset with PCOS_data saved as 'merged_aggregated_womens_health_with_pcos.csv'.
   age_bin  CycleNumber     Group  CycleWithPeakorNot  ReproductiveCategory  \
0     15.0          NaN       NaN                 NaN                   NaN   
1     20.0     7.758197  0.370902            0.899590              0.084016   
2     25.0     7.090090  0.417417            0.912913              0.099099   
3     30.0     7.313924  0.412658            0.916456              0.030380   
4     35.0    10.074813  0.294264            0.937656              0.014963   

   LengthofCycle      new_id        age  cycle_number  cycle_length  ...  \
0            NaN  163.500000  18.722222      4.722222           0.0  ...   
1      29.973361  314.686869  23.266511      6.724165           0.0  ...   
2      29.507508  279.141994  27.794562      6.028701           0.0  ...   
3      29.673418  266.882784  32.619048      6.126374           0.0  ...   
4      28.059850  357.778523  38.127517      6